In [1]:
import numpy as np
import pandas as pd
from pyhive import presto
from datetime import timedelta,datetime
import json
import requests
import math
import os


pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings("ignore")
conn = presto.connect(
        host="presto-gateway.serving.data.plectrum.dev",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )
presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

In [5]:
ferrao_query = f"""select
    date_format(date_trunc('week', date_parse(day, '%Y-%m-%d')), '%Y%m%d') week_start_date,
    customerid,
    sum(ao_session) ao_session,
    sum(fe_session) fe_session,
    sum(rr_session) rr_session,
    sum(ao_day) ao_day,
    sum(fe_day) fe_day,
    sum(rr_day) rr_day
from
(
    select 
        day,
        customerid, 
        max(ao_sessions_unique_daily) ao_session, 
        max(fe_sessions_unique_daily) fe_session, 
        sum(rr_sessions_unique_daily) rr_session, 
        max(case when ao_sessions_unique_daily > 0 then 1 else 0 end) ao_day,
        max(case when fe_sessions_unique_daily > 0 then 1 else 0 end) fe_day,
        max(case when rr_sessions_unique_daily > 0 then 1 else 0 end) rr_day
    from 
        datasets.customer_rf_daily_kpi
    where 
        day >= '2024-12-16'
        and day <= '2025-03-16'
        and city = 'Bangalore'
        and service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    order by 1
)
group by 1,2"""

In [9]:
ferrao_df = pd.read_sql(ferrao_query, conn3)

In [13]:
ferrao_df.to_csv('ferrao_df.csv', index=False, header=False)

In [17]:
ferrao_df.columns

Index(['week_start_date', 'customerid', 'ao_session', 'fe_session',
       'rr_session', 'ao_day', 'fe_day', 'rr_day'],
      dtype='object')

In [151]:
ferrao_df['rr_day'].sum()

34063611

In [115]:
ferrao_query_link = f"""select
    date_format(date_trunc('week', date_parse(day, '%Y-%m-%d')), '%Y%m%d') week_start_date,
    customerid,
    sum(ao_session) ao_session,
    sum(fe_session) fe_session,
    sum(rr_session) rr_session,
    sum(ao_day) ao_day,
    sum(fe_day) fe_day,
    sum(rr_day) rr_day
from
(
    select 
        day,
        customerid, 
        max(ao_sessions_unique_daily) ao_session, 
        max(fe_sessions_unique_daily) fe_session, 
        sum(rr_sessions_unique_daily) rr_session, 
        max(case when ao_sessions_unique_daily > 0 then 1 else 0 end) ao_day,
        max(case when fe_sessions_unique_daily > 0 then 1 else 0 end) fe_day,
        max(case when rr_sessions_unique_daily > 0 then 1 else 0 end) rr_day
    from 
        datasets.customer_rf_daily_kpi
    where 
        day >= '2024-12-16'
        and day <= '2025-03-16'
        and city = 'Bangalore'
        and service_name in ('Link','Bike Lite','Bike Pink','Bike Metro')
    group by 1,2
    order by 1
)
group by 1,2"""

In [117]:
ferrao_df_link = pd.read_sql(ferrao_query_link, conn4)

In [123]:
ferrao_df_link

,week_start_date,customerid,ao_session,fe_session,rr_session,ao_day,fe_day,rr_day
0,20250303,677ffa9b57212b40a39ec98f,16,11,0,5,5,0
1,20250303,61e29e6f0fe854136e861dce,0,0,0,0,0,0
2,20250127,666561d47000d9a6c0368a91,12,9,0,7,7,0
3,20241223,67540a8364d2b768ae5981da,1,1,0,1,1,0
4,20250210,5cf77241c0b26018f5d9a417,4,3,3,3,3,3
...,...,...,...,...,...,...,...,...
23755233,20250203,67a6240cc566d456466e7845,2,2,1,1,1,1
23755234,20241216,5dcea95d0856f146182f170e,2,1,0,2,1,0
23755235,20241230,62bd5155b00b293f6bc0e535,1,1,0,1,1,0
23755236,20241216,636f5fd783ee5363223b791f,1,1,1,1,1,1


In [127]:
ferrao_df_link.to_csv('ferrao_df_link.csv', index=False, header=False)

In [145]:
ferrao_df_link['rr_day'].sum()

12048089

In [129]:
ferrao_query_auto = f"""select
    date_format(date_trunc('week', date_parse(day, '%Y-%m-%d')), '%Y%m%d') week_start_date,
    customerid,
    sum(ao_session) ao_session,
    sum(fe_session) fe_session,
    sum(rr_session) rr_session,
    sum(ao_day) ao_day,
    sum(fe_day) fe_day,
    sum(rr_day) rr_day
from
(
    select 
        day,
        customerid, 
        max(ao_sessions_unique_daily) ao_session, 
        max(fe_sessions_unique_daily) fe_session, 
        sum(rr_sessions_unique_daily) rr_session, 
        max(case when ao_sessions_unique_daily > 0 then 1 else 0 end) ao_day,
        max(case when fe_sessions_unique_daily > 0 then 1 else 0 end) fe_day,
        max(case when rr_sessions_unique_daily > 0 then 1 else 0 end) rr_day
    from 
        datasets.customer_rf_daily_kpi
    where 
        day >= '2024-12-16'
        and day <= '2025-03-16'
        and city = 'Bangalore'
        and service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium')
    group by 1,2
    order by 1
)
group by 1,2"""

In [131]:
ferrao_df_auto = pd.read_sql(ferrao_query_auto, conn4)

In [132]:
ferrao_df_auto

,week_start_date,customerid,ao_session,fe_session,rr_session,ao_day,fe_day,rr_day
0,20250310,626188631b5711007e82722c,6,6,6,6,6,6
1,20241223,5d99eba3a1f0a71c605bbfea,3,3,1,3,3,1
2,20250106,5d119d9bd1212c6a896cecf1,3,2,1,1,1,1
3,20250106,62e3f427cb61573ea55da599,3,1,0,1,1,0
4,20241216,645ecb33b3d0c2630ddc898b,2,2,1,2,2,1
...,...,...,...,...,...,...,...,...
23770662,20250224,64e238db7284de0245c84d36,1,1,0,1,1,0
23770663,20250303,648d8dcfc0b55439137f772c,0,0,0,0,0,0
23770664,20250203,656b199c2251f0365947bcf9,3,1,1,1,1,1
23770665,20250106,5f469ed808f2ce74a351a5f5,0,0,0,0,0,0


In [155]:
ferrao_df_auto.to_csv('ferrao_df_auto.csv', index=False, header=False)

In [147]:
ferrao_df_auto['rr_day'].sum()

21261017

In [153]:
21261017+12048089

33309106

In [ ]:
34063611

### new segment

In [27]:
cx_data0 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    
    select
        segment,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
       
'''
cx_data0 = pd.read_sql(cx_data0, conn4)
cx_data0.head()

,segment,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,1-DAILY,1201195,951737,1032303,1371984,1272490,1436490,1414093,1477300,1517082,1418385,1362470,1413752,1324677,17193958,18.41,912886,732507,796769,1036457,966282,1084869,1076156,1114153,1125374,1062090,1022569,1056792,992102,12979006,17.87,800419,626304,685139,904965,845143,961013,944157,993429,1003560,932908,909194,944933,877971,11429135,22.57
1,2-WEEKLY,1405959,1165313,1270516,1568417,1466900,1604353,1574133,1674840,1735113,1599054,1559568,1606208,1547498,19777872,21.18,1078948,905298,986833,1198507,1125905,1227234,1210491,1277935,1306796,1210473,1184898,1216190,1174630,15104138,20.80,848443,686187,749978,932049,876354,974733,951052,1021220,1044353,949628,940549,979767,935157,11889470,23.48
2,3-MONTHLY,3058159,2654450,2846046,3323344,3302530,3522341,3533399,3996920,3982207,3517254,3430567,3483332,3536305,44186854,47.31,2417845,2119148,2267344,2614056,2611415,2774234,2787985,3129559,3094037,2744072,2687412,2717327,2758167,34722601,47.82,1584602,1317511,1415076,1662712,1657044,1805442,1797039,2078094,2048117,1776444,1766202,1835862,1843816,22587961,44.61
3,4-OTHER,783689,683712,652393,596536,566100,630948,629135,651673,943551,1094488,1181766,1343105,1473987,11231083,12.02,625338,551165,524970,479609,457342,505380,505264,523308,737982,855498,926519,1046590,1147824,8886789,12.24,340506,282122,263683,228132,209526,246753,241881,249871,391044,463226,519363,614845,677560,4728512,9.34
4,5-INACTIVE,97343,83143,79743,78945,74886,72410,71212,74181,76529,73421,72077,72842,85212,1011944,1.08,89886,76792,73530,72287,68677,66555,65348,68343,70096,66927,66139,65160,73796,923536,1.27,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.00


In [11]:
cx_data0

,segment,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,1-DAILY,631610,519804,560675,707363,656647,725695,714072,743633,765160,720194,696737,716639,676586,8834815,9.46,481647,401225,433228,535441,500142,549631,544760,561614,568228,540375,523595,536382,508099,6684367,9.21,430825,352086,382348,478313,447940,497585,489020,511335,517020,484979,475622,488817,458422,6014312,11.88
1,2-WEEKLY,1958837,1582800,1727425,2212689,2055922,2277315,2234585,2367186,2447785,2261856,2191505,2269058,2163654,27750617,29.71,1497203,1225299,1338743,1683729,1571564,1733785,1711533,1798936,1834174,1705413,1658229,1710822,1634389,21103819,29.06,1206656,950591,1042621,1345162,1256199,1413110,1379862,1475378,1504429,1374050,1351141,1412577,1333008,17044784,33.66
2,3-MONTHLY,3074866,2668896,2860765,3343693,3329351,3560174,3572968,4038241,4021457,3552643,3464363,3517595,3568240,44573252,47.72,2430829,2130429,2278975,2629850,2631896,2802921,2818339,3161097,3123805,2770847,2713055,2743105,2782411,35017559,48.22,1595983,1327325,1425224,1676251,1674402,1830493,1823366,2106030,2074581,1799951,1789182,1859168,1865514,22847470,45.12
3,4-OTHER,783689,683712,652393,596536,566100,630948,629135,651673,943551,1094488,1181766,1343105,1473987,11231083,12.02,625338,551165,524970,479609,457342,505380,505264,523308,737982,855498,926519,1046590,1147824,8886789,12.24,340506,282122,263683,228132,209526,246753,241881,249871,391044,463226,519363,614845,677560,4728512,9.34
4,5-INACTIVE,97343,83143,79743,78945,74886,72410,71212,74181,76529,73421,72077,72842,85212,1011944,1.08,89886,76792,73530,72287,68677,66555,65348,68343,70096,66927,66139,65160,73796,923536,1.27,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.00


In [28]:
cx_data0.to_clipboard(index=False)

### Office bins

In [2]:
cx_data15 = f'''
    with base as (
    
    select
        customer_id,
        segment,
        -- total_net_orders,
        -- office_orders,
        /*
        case 
        when office_orders = '0' then 'a. 0 office ride'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        else 'a. 0 office ride'
        end office_bin
    
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        when transit_orders = '1' then 'g. 1 transit ride'
        when transit_orders >= '2' and transit_orders <= '3' then 'h. 2-3 transit ride'
        when transit_orders > '3' then 'i. 3+ transit ride'
        else 'a. no-office-transit'
        end office_bin
        */
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders >= '1' then 'b. office ride'
        when transit_orders >= '1' then 'c. transit ride'
        else 'a. no-office-transit'
        end office_bin
        
        
    from 
        hive.experiments_internal.customer_base_office_transit_tag_amj2025
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    
    select
        office_bin,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
       
'''
cx_data15 = pd.read_sql(cx_data15, conn3)
cx_data15.head()

,office_bin,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,a. no-office-transit,1860844,1628151,1695217,1849773,1834836,1918437,1911578,2120646,2233114,2085712,2081218,2160982,2313032,25693540,27.51,1503044,1322533,1373741,1488038,1479505,1543716,1539759,1698079,1771423,1659894,1659619,1716389,1831503,20587243,28.35,879151,733742,770534,853042,848828,909428,897937,1018435,1058483,968895,984058,1045694,1105075,12073302,23.84
1,b. office ride,3725964,3039915,3261401,4078057,3837376,4298142,4264811,4615117,4830479,4498697,4418107,4629831,4515350,54013247,57.83,2856723,2363665,2535601,3108109,2944106,3279393,3271856,3510929,3625853,3397868,3351841,3495978,3415555,41157477,56.68,2184393,1729817,1863549,2348226,2211655,2522047,2485947,2715463,2803723,2575568,2569629,2726116,2624942,31361075,61.94
2,c. transit ride,959537,870289,924383,1011396,1010694,1049963,1045583,1139151,1190889,1118193,1107123,1128426,1139297,13694924,14.66,765136,698712,740104,804769,806010,835163,833629,904290,937009,881298,876077,889692,899461,10871350,14.97,510426,448565,479793,526590,527584,556466,550245,608716,624868,577743,581621,603597,604487,7200701,14.22


In [3]:
cx_data15

,office_bin,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,a. no-office-transit,1860844,1628151,1695217,1849773,1834836,1918437,1911578,2120646,2233114,2085712,2081218,2160982,2313032,25693540,27.51,1503044,1322533,1373741,1488038,1479505,1543716,1539759,1698079,1771423,1659894,1659619,1716389,1831503,20587243,28.35,879151,733742,770534,853042,848828,909428,897937,1018435,1058483,968895,984058,1045694,1105075,12073302,23.84
1,b. office ride,3725964,3039915,3261401,4078057,3837376,4298142,4264811,4615117,4830479,4498697,4418107,4629831,4515350,54013247,57.83,2856723,2363665,2535601,3108109,2944106,3279393,3271856,3510929,3625853,3397868,3351841,3495978,3415555,41157477,56.68,2184393,1729817,1863549,2348226,2211655,2522047,2485947,2715463,2803723,2575568,2569629,2726116,2624942,31361075,61.94
2,c. transit ride,959537,870289,924383,1011396,1010694,1049963,1045583,1139151,1190889,1118193,1107123,1128426,1139297,13694924,14.66,765136,698712,740104,804769,806010,835163,833629,904290,937009,881298,876077,889692,899461,10871350,14.97,510426,448565,479793,526590,527584,556466,550245,608716,624868,577743,581621,603597,604487,7200701,14.22


In [6]:
cx_data15.to_clipboard(index=False)

In [4]:
cx_data16 = f'''
    with base as (
    
    select
        customer_id,
        segment,
        -- total_net_orders,
        -- office_orders,
        /*
        case 
        when office_orders = '0' then 'a. 0 office ride'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        else 'a. 0 office ride'
        end office_bin
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        when transit_orders = '1' then 'g. 1 transit ride'
        when transit_orders >= '2' and transit_orders <= '3' then 'h. 2-3 transit ride'
        when transit_orders > '3' then 'i. 3+ transit ride'
        else 'a. no-office-transit'
        end office_bin
        */
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders >= '1' then 'b. office ride'
        when transit_orders >= '1' then 'c. transit ride'
        else 'a. no-office-transit'
        end office_bin
        
    from 
        hive.experiments_internal.customer_base_office_transit_tag_amj2025
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    
    select
        segment,
        office_bin,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1,2
    order by 1,2
       
'''
cx_data16 = pd.read_sql(cx_data16, conn3)
cx_data16.head()

,segment,office_bin,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,1-DAILY,a. no-office-transit,100203,77120,87121,115141,106998,123185,120836,125256,126932,119664,112401,117691,110977,1443525,1.55,79319,61482,69881,91528,85137,97704,96452,99266,99243,94357,88367,92985,87484,1143205,1.57,69139,52384,59719,79046,74054,85687,83998,87074,87211,81862,77882,82005,76569,996630,1.97
1,1-DAILY,b. office ride,998140,790129,850833,1137978,1054503,1188409,1170379,1223591,1259616,1175010,1133503,1174102,1098466,14254659,15.26,752081,603667,651344,851149,793609,888570,882567,913711,924308,871027,842956,868847,814181,10658017,14.68,661159,517157,561227,745684,695623,789552,776319,818209,827952,767499,751658,779470,722694,9414203,18.59
2,1-DAILY,c. transit ride,102852,84488,94349,118865,110989,124896,122878,128453,130534,123711,116566,121959,115234,1495774,1.60,81486,67358,75544,93780,87536,98595,97137,101176,101823,96706,91246,94960,90437,1177784,1.62,70121,56763,64193,80235,75466,85774,83840,88146,88397,83547,79654,83458,78708,1018302,2.01
3,2-WEEKLY,a. no-office-transit,188516,158556,171462,207057,196660,210579,205459,219877,225003,209630,201467,205512,202535,2602313,2.79,148379,125321,135847,162795,154595,165774,162462,172972,174910,163605,157267,160656,158139,2042722,2.81,114594,93484,101884,123854,118814,128979,125359,135079,136117,125740,122357,126736,123676,1576673,3.11
4,2-WEEKLY,b. office ride,1040057,848585,925371,1164707,1080069,1193103,1171163,1245763,1293501,1189819,1161859,1202292,1151155,14667444,15.70,790632,654250,713275,880462,821195,903526,891753,940508,963034,890868,873554,900391,865136,11088584,15.27,626785,499491,545129,691202,643521,724382,706364,758958,778565,705709,700094,732547,694861,8807608,17.39


In [7]:
cx_data16

,segment,office_bin,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,1-DAILY,a. no-office-transit,100203,77120,87121,115141,106998,123185,120836,125256,126932,119664,112401,117691,110977,1443525,1.55,79319,61482,69881,91528,85137,97704,96452,99266,99243,94357,88367,92985,87484,1143205,1.57,69139,52384,59719,79046,74054,85687,83998,87074,87211,81862,77882,82005,76569,996630,1.97
1,1-DAILY,b. office ride,998140,790129,850833,1137978,1054503,1188409,1170379,1223591,1259616,1175010,1133503,1174102,1098466,14254659,15.26,752081,603667,651344,851149,793609,888570,882567,913711,924308,871027,842956,868847,814181,10658017,14.68,661159,517157,561227,745684,695623,789552,776319,818209,827952,767499,751658,779470,722694,9414203,18.59
2,1-DAILY,c. transit ride,102852,84488,94349,118865,110989,124896,122878,128453,130534,123711,116566,121959,115234,1495774,1.60,81486,67358,75544,93780,87536,98595,97137,101176,101823,96706,91246,94960,90437,1177784,1.62,70121,56763,64193,80235,75466,85774,83840,88146,88397,83547,79654,83458,78708,1018302,2.01
3,2-WEEKLY,a. no-office-transit,188516,158556,171462,207057,196660,210579,205459,219877,225003,209630,201467,205512,202535,2602313,2.79,148379,125321,135847,162795,154595,165774,162462,172972,174910,163605,157267,160656,158139,2042722,2.81,114594,93484,101884,123854,118814,128979,125359,135079,136117,125740,122357,126736,123676,1576673,3.11
4,2-WEEKLY,b. office ride,1040057,848585,925371,1164707,1080069,1193103,1171163,1245763,1293501,1189819,1161859,1202292,1151155,14667444,15.70,790632,654250,713275,880462,821195,903526,891753,940508,963034,890868,873554,900391,865136,11088584,15.27,626785,499491,545129,691202,643521,724382,706364,758958,778565,705709,700094,732547,694861,8807608,17.39
5,2-WEEKLY,c. transit ride,177386,158172,173683,196653,190171,200671,197511,209200,216609,199605,196242,198404,193808,2508115,2.69,139937,125727,137711,155250,150115,157934,156276,164455,168852,156000,154077,155143,151355,1972832,2.72,107064,93212,102965,116993,114019,121372,119329,127183,129671,118179,118098,120484,116620,1505189,2.97
6,3-MONTHLY,a. no-office-transit,996679,891852,952173,1073410,1090491,1117467,1119747,1282167,1253195,1086498,1057084,1053434,1098002,14072199,15.07,800424,719948,766511,857033,873544,894549,896154,1018866,991107,862962,842121,836245,870933,11230397,15.47,494712,422922,451402,511347,523558,545258,541580,638093,615130,522038,517810,528381,547393,6859624,13.55
7,3-MONTHLY,b. office ride,1504890,1247324,1341268,1647220,1591005,1777700,1784542,2010734,2030132,1808662,1762079,1823437,1822502,22151495,23.72,1171727,984010,1057141,1275998,1240733,1378827,1388589,1550196,1550205,1387990,1359100,1399579,1398352,17142447,23.61,808795,642607,692884,857577,828296,948634,944943,1083225,1084599,949940,942761,996901,983920,11765082,23.24
8,3-MONTHLY,c. transit ride,556590,515274,552605,602714,621034,627174,629110,704019,698880,622094,611404,606461,615801,7963160,8.53,445694,415190,443692,481025,497138,500858,503242,560497,552725,493120,486191,481503,488882,6349757,8.74,281095,251982,270790,293788,305190,311550,310516,356776,348388,304466,305631,310580,312503,3963255,7.83
9,4-OTHER,a. no-office-transit,479529,418753,405826,375992,366696,395738,395306,420302,552681,597681,639435,712874,818008,6578821,7.04,386258,340064,328936,305075,298333,319962,320212,339523,437030,472995,506758,562396,642384,5259926,7.24,200706,164952,157529,138795,132402,149504,147000,158189,220025,239255,266009,308572,357437,2640375,5.21


In [8]:
cx_data16.to_clipboard(index=False)

In [13]:
cx_data01 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /*case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  */
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check

        /*CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check*/

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        coalesce(RHA_Check, 'UNKNOWN') RHA_Check,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data11 = pd.read_sql(cx_data01, conn1)
cx_data11.head()

,segment,RHA_Check,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,1-DAILY,NRHA,70421,65619,70053,80511,76959,82251,81123,85173,87069,82647,79384,80705,76460,1018375,1.09,56299,52366,55887,63779,61104,65485,64623,67447,68498,65244,62854,63760,60738,808084,1.11,50170,46029,49680,56980,55211,59257,57962,61338,62454,58670,57235,58246,54882,728114,1.44
1,1-DAILY,RHA,349720,292966,317787,390067,362165,401258,395661,408661,417975,396675,382578,394385,372858,4882756,5.23,288161,241635,261993,319292,297910,328094,325067,335281,337923,322508,311916,319640,303496,3992916,5.50,258416,212215,231655,285754,266573,297902,292084,305218,306855,288184,282578,290741,272850,3591025,7.09
2,1-DAILY,UNKNOWN,211469,161219,172835,236785,217523,242186,237288,249799,260116,240872,234775,241549,227268,2933684,3.14,137187,107224,115348,152370,141128,156052,155070,158886,161807,152623,148825,152982,143865,1883367,2.59,122239,93842,101013,135579,126156,140426,138974,144779,147711,138125,135809,139830,130690,1695173,3.35
3,2-WEEKLY,NRHA,250118,233655,253546,288472,276625,292297,287362,302065,310753,288722,279285,283428,273615,3619943,3.88,199431,186371,202039,229166,219905,232282,229032,240890,245565,228834,221631,224163,216917,2876226,3.96,157507,143256,156432,179202,172597,184356,181157,193469,196337,180464,176885,181207,173432,2276301,4.50
4,2-WEEKLY,RHA,1084623,887409,970829,1216555,1135647,1253215,1235137,1299744,1337728,1246267,1211667,1248440,1192729,15319990,16.40,894227,733985,802964,999883,934655,1030430,1018162,1069713,1088855,1017033,990542,1019143,974777,12574369,17.32,719628,567372,623639,796966,744261,838309,818306,874495,888570,813250,801876,836920,789773,10113365,19.97


In [14]:
cx_data11

,segment,RHA_Check,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,1-DAILY,NRHA,70421,65619,70053,80511,76959,82251,81123,85173,87069,82647,79384,80705,76460,1018375,1.09,56299,52366,55887,63779,61104,65485,64623,67447,68498,65244,62854,63760,60738,808084,1.11,50170,46029,49680,56980,55211,59257,57962,61338,62454,58670,57235,58246,54882,728114,1.44
1,1-DAILY,RHA,349720,292966,317787,390067,362165,401258,395661,408661,417975,396675,382578,394385,372858,4882756,5.23,288161,241635,261993,319292,297910,328094,325067,335281,337923,322508,311916,319640,303496,3992916,5.50,258416,212215,231655,285754,266573,297902,292084,305218,306855,288184,282578,290741,272850,3591025,7.09
2,1-DAILY,UNKNOWN,211469,161219,172835,236785,217523,242186,237288,249799,260116,240872,234775,241549,227268,2933684,3.14,137187,107224,115348,152370,141128,156052,155070,158886,161807,152623,148825,152982,143865,1883367,2.59,122239,93842,101013,135579,126156,140426,138974,144779,147711,138125,135809,139830,130690,1695173,3.35
3,2-WEEKLY,NRHA,250118,233655,253546,288472,276625,292297,287362,302065,310753,288722,279285,283428,273615,3619943,3.88,199431,186371,202039,229166,219905,232282,229032,240890,245565,228834,221631,224163,216917,2876226,3.96,157507,143256,156432,179202,172597,184356,181157,193469,196337,180464,176885,181207,173432,2276301,4.50
4,2-WEEKLY,RHA,1084623,887409,970829,1216555,1135647,1253215,1235137,1299744,1337728,1246267,1211667,1248440,1192729,15319990,16.40,894227,733985,802964,999883,934655,1030430,1018162,1069713,1088855,1017033,990542,1019143,974777,12574369,17.32,719628,567372,623639,796966,744261,838309,818306,874495,888570,813250,801876,836920,789773,10113365,19.97
5,2-WEEKLY,UNKNOWN,624096,461736,503050,707662,643650,731803,712086,765377,799304,726867,700553,737190,697310,8810684,9.43,403545,304943,333740,454680,417004,471073,464339,488333,499754,459546,446056,467516,442695,5653224,7.79,329521,239963,262550,368994,339341,390445,380399,407414,419522,380336,372380,394450,369803,4655118,9.19
6,3-MONTHLY,NRHA,696519,668461,726083,812717,833754,825424,826977,936059,913453,797208,779034,771963,803957,10391609,11.13,559294,537622,581460,647667,666246,660180,661876,747659,726643,636514,623684,616250,640257,8305352,11.44,346070,320871,349019,388632,400299,400746,397697,464069,449195,386605,384210,390005,403478,5080896,10.03
7,3-MONTHLY,RHA,1677590,1458640,1553392,1788529,1787406,1924862,1932930,2163464,2157071,1921273,1885812,1909224,1924863,24085056,25.79,1400924,1222226,1299957,1488131,1490050,1601992,1609563,1797029,1779852,1588618,1564138,1578890,1593494,20014864,27.56,917146,758581,809159,945575,944918,1045319,1040299,1196266,1178940,1025258,1024987,1064907,1061314,13012669,25.70
8,3-MONTHLY,UNKNOWN,700757,541795,581290,742447,708191,809888,813061,938718,950933,834162,799517,836408,839420,10096587,10.81,470611,370581,397558,494052,475600,540749,546900,616409,617310,545715,525233,547965,548660,6697343,9.22,332767,247873,267046,342044,329185,384428,385370,445695,446446,388088,379985,404256,400722,4753905,9.39
9,4-OTHER,NRHA,258113,235541,228732,211822,206312,219180,219537,225822,317388,363291,388527,436286,486820,3797371,4.07,205040,188545,182770,168967,164779,174509,174764,180810,248529,284777,306019,342317,380603,3002429,4.13,105895,92915,88758,77952,73297,81057,79457,82042,121825,142446,157902,183685,205782,1493013,2.95


In [17]:
cx_data11.to_clipboard(index=False)

In [15]:
cx_data02 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        */

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        coalesce(Zwigato_check, 'UNKNOWN') Zwigato_check,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data12 = pd.read_sql(cx_data02, conn1)
cx_data12.head()

,segment,Zwigato_check,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,1-DAILY,Non-Zwigato,94534,86717,92445,107656,101562,110847,109400,113497,115922,110746,105315,107092,101305,1357038,1.45,75263,69448,73777,85353,80867,88031,87205,90048,91175,87413,83243,84412,80162,1076397,1.48,64742,59276,63462,73532,70428,77002,75757,79043,80001,75275,73117,74332,69760,935727,1.85
1,1-DAILY,UNKNOWN,211469,161219,172835,236785,217523,242186,237288,249799,260116,240872,234775,241549,227268,2933684,3.14,137187,107224,115348,152370,141128,156052,155070,158886,161807,152623,148825,152982,143865,1883367,2.59,122239,93842,101013,135579,126156,140426,138974,144779,147711,138125,135809,139830,130690,1695173,3.35
2,1-DAILY,Zwigato,325607,271868,295395,362922,337562,372662,367384,380337,389122,368576,356647,367998,348013,4544093,4.87,269197,224553,244103,297718,278147,305548,302485,312680,315246,300339,291527,298988,284072,3724603,5.13,243844,198968,217873,269202,251356,280157,274289,287513,289308,271579,266696,274655,257972,3383412,6.68
3,2-WEEKLY,Non-Zwigato,321275,294441,317001,367568,351439,373860,370452,388470,398005,372063,356180,362106,346856,4619716,4.95,256257,235571,253634,293019,280472,298436,296043,310733,315306,295694,283241,286932,275630,3680968,5.07,194664,174261,188733,219847,211060,227749,224339,238640,241644,222668,216755,221941,210751,2793052,5.52
4,2-WEEKLY,UNKNOWN,624096,461736,503050,707662,643650,731803,712086,765377,799304,726867,700553,737190,697310,8810684,9.43,403545,304943,333740,454680,417004,471073,464339,488333,499754,459546,446056,467516,442695,5653224,7.79,329521,239963,262550,368994,339341,390445,380399,407414,419522,380336,372380,394450,369803,4655118,9.19


In [16]:
cx_data12

,segment,Zwigato_check,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,1-DAILY,Non-Zwigato,94534,86717,92445,107656,101562,110847,109400,113497,115922,110746,105315,107092,101305,1357038,1.45,75263,69448,73777,85353,80867,88031,87205,90048,91175,87413,83243,84412,80162,1076397,1.48,64742,59276,63462,73532,70428,77002,75757,79043,80001,75275,73117,74332,69760,935727,1.85
1,1-DAILY,UNKNOWN,211469,161219,172835,236785,217523,242186,237288,249799,260116,240872,234775,241549,227268,2933684,3.14,137187,107224,115348,152370,141128,156052,155070,158886,161807,152623,148825,152982,143865,1883367,2.59,122239,93842,101013,135579,126156,140426,138974,144779,147711,138125,135809,139830,130690,1695173,3.35
2,1-DAILY,Zwigato,325607,271868,295395,362922,337562,372662,367384,380337,389122,368576,356647,367998,348013,4544093,4.87,269197,224553,244103,297718,278147,305548,302485,312680,315246,300339,291527,298988,284072,3724603,5.13,243844,198968,217873,269202,251356,280157,274289,287513,289308,271579,266696,274655,257972,3383412,6.68
3,2-WEEKLY,Non-Zwigato,321275,294441,317001,367568,351439,373860,370452,388470,398005,372063,356180,362106,346856,4619716,4.95,256257,235571,253634,293019,280472,298436,296043,310733,315306,295694,283241,286932,275630,3680968,5.07,194664,174261,188733,219847,211060,227749,224339,238640,241644,222668,216755,221941,210751,2793052,5.52
4,2-WEEKLY,UNKNOWN,624096,461736,503050,707662,643650,731803,712086,765377,799304,726867,700553,737190,697310,8810684,9.43,403545,304943,333740,454680,417004,471073,464339,488333,499754,459546,446056,467516,442695,5653224,7.79,329521,239963,262550,368994,339341,390445,380399,407414,419522,380336,372380,394450,369803,4655118,9.19
5,2-WEEKLY,Zwigato,1013466,826623,907374,1137459,1060833,1171652,1152047,1213339,1250476,1162926,1134772,1169762,1119488,14320217,15.33,837401,684785,751369,936030,874088,964276,951151,999870,1019114,950173,928932,956374,916064,11769627,16.21,682471,536367,591338,756321,705798,794916,775124,829324,843263,771046,762006,796186,752454,9596614,18.95
6,3-MONTHLY,Non-Zwigato,857632,820662,877585,985758,1018715,1022570,1022803,1147131,1119664,978934,942830,936605,965522,12696411,13.59,691486,663454,706814,791146,819964,821599,822570,920882,895208,784959,757419,750746,772916,10199163,14.05,416934,385734,411628,462563,479087,486917,483248,558487,539340,463586,453626,461158,472466,6074774,12.00
7,3-MONTHLY,UNKNOWN,700757,541795,581290,742447,708191,809888,813061,938718,950933,834162,799517,836408,839420,10096587,10.81,470611,370581,397558,494052,475600,540749,546900,616409,617310,545715,525233,547965,548660,6697343,9.22,332767,247873,267046,342044,329185,384428,385370,445695,446446,388088,379985,404256,400722,4753905,9.39
8,3-MONTHLY,Zwigato,1516477,1306439,1401890,1615488,1602445,1727716,1737104,1952392,1950860,1739547,1722016,1744582,1763298,21780254,23.32,1268732,1096394,1174603,1344652,1336332,1440573,1448869,1623806,1611287,1440173,1430403,1444394,1460835,18121053,24.95,846282,693718,746550,871644,866130,959148,954748,1101848,1088795,948277,955571,993754,992326,12018791,23.74
9,4-OTHER,Non-Zwigato,322479,294416,277686,256610,251934,270393,268287,274739,381752,437066,465627,516420,576519,4593928,4.92,257770,237075,223536,205738,202345,216054,214869,220709,299824,343489,367725,407091,452418,3648643,5.02,132461,115659,106352,93142,87869,98649,96088,98733,145529,169974,187973,216595,242841,1791865,3.54


In [18]:
cx_data12.to_clipboard(index=False)

In [19]:
cx_data03 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        case 
        when RHA_Check = 'RHA' and Zwigato_check = 'Zwigato' then '1. BOTH'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Zwigato' then '2. ONLY Zwigato'
        when RHA_Check = 'RHA' and Zwigato_check = 'Non-Zwigato' then '3. ONLY RHA'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Non-Zwigato' then '4. NONE'
        ELSE '5. IOS' end tech_savvy,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data13 = pd.read_sql(cx_data03, conn1)
cx_data13.head()

,segment,tech_savvy,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,1-DAILY,1. BOTH,292632,241730,262924,326030,301966,335050,330236,341361,349332,330400,319895,330247,312406,4074209,4.36,242310,200174,217776,267811,249435,275028,272314,281017,283292,269637,261824,268645,255202,3344465,4.61,219320,177051,193995,241880,224903,251967,246691,258159,259743,243462,239176,246483,231485,3034315,5.99
1,1-DAILY,2. ONLY Zwigato,32975,30138,32471,36892,35596,37612,37148,38976,39790,38176,36752,37751,35607,469884,0.50,26887,24379,26327,29907,28712,30520,30171,31663,31954,30702,29703,30343,28870,380138,0.52,24524,21917,23878,27322,26453,28190,27598,29354,29565,28117,27520,28172,26487,349097,0.69
2,1-DAILY,3. ONLY RHA,57088,51236,54863,64037,60199,66208,65425,67300,68643,66275,62683,64138,60452,808547,0.87,45851,41461,44217,51481,48475,53066,52753,54264,54631,52871,50092,50995,48294,648451,0.89,39096,35164,37660,43874,41670,45935,45393,47059,47112,44722,43402,44258,41365,556710,1.10
3,1-DAILY,4. NONE,37446,35481,37582,43619,41363,44639,43975,46197,47279,44471,42632,42954,40853,548491,0.59,29412,27987,29560,33872,32392,34965,34452,35784,36544,34542,33151,33417,31868,427946,0.59,25646,24112,25802,29658,28758,31067,30364,31984,32889,30553,29715,30074,28395,379017,0.75
4,1-DAILY,5. IOS,211469,161219,172835,236785,217523,242186,237288,249799,260116,240872,234775,241549,227268,2933684,3.14,137187,107224,115348,152370,141128,156052,155070,158886,161807,152623,148825,152982,143865,1883367,2.59,122239,93842,101013,135579,126156,140426,138974,144779,147711,138125,135809,139830,130690,1695173,3.35


In [20]:
cx_data13

,segment,tech_savvy,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,1-DAILY,1. BOTH,292632,241730,262924,326030,301966,335050,330236,341361,349332,330400,319895,330247,312406,4074209,4.36,242310,200174,217776,267811,249435,275028,272314,281017,283292,269637,261824,268645,255202,3344465,4.61,219320,177051,193995,241880,224903,251967,246691,258159,259743,243462,239176,246483,231485,3034315,5.99
1,1-DAILY,2. ONLY Zwigato,32975,30138,32471,36892,35596,37612,37148,38976,39790,38176,36752,37751,35607,469884,0.50,26887,24379,26327,29907,28712,30520,30171,31663,31954,30702,29703,30343,28870,380138,0.52,24524,21917,23878,27322,26453,28190,27598,29354,29565,28117,27520,28172,26487,349097,0.69
2,1-DAILY,3. ONLY RHA,57088,51236,54863,64037,60199,66208,65425,67300,68643,66275,62683,64138,60452,808547,0.87,45851,41461,44217,51481,48475,53066,52753,54264,54631,52871,50092,50995,48294,648451,0.89,39096,35164,37660,43874,41670,45935,45393,47059,47112,44722,43402,44258,41365,556710,1.10
3,1-DAILY,4. NONE,37446,35481,37582,43619,41363,44639,43975,46197,47279,44471,42632,42954,40853,548491,0.59,29412,27987,29560,33872,32392,34965,34452,35784,36544,34542,33151,33417,31868,427946,0.59,25646,24112,25802,29658,28758,31067,30364,31984,32889,30553,29715,30074,28395,379017,0.75
4,1-DAILY,5. IOS,211469,161219,172835,236785,217523,242186,237288,249799,260116,240872,234775,241549,227268,2933684,3.14,137187,107224,115348,152370,141128,156052,155070,158886,161807,152623,148825,152982,143865,1883367,2.59,122239,93842,101013,135579,126156,140426,138974,144779,147711,138125,135809,139830,130690,1695173,3.35
5,2-WEEKLY,1. BOTH,901808,724705,796193,1010472,939684,1042176,1025085,1079206,1113371,1035507,1010739,1043753,996192,12718891,13.62,747138,602133,661410,833693,776778,860185,848724,891472,909400,847938,829400,855334,817194,10480799,14.43,609251,470948,519428,673638,626703,709697,691621,739381,752749,687601,680304,711927,670802,8544050,16.87
6,2-WEEKLY,2. ONLY Zwigato,111658,101918,111181,126987,121149,129476,126962,134133,137105,127419,124033,126009,123296,1601326,1.71,90263,82652,89959,102337,97310,104091,102427,108398,109714,102235,99532,101040,98870,1288828,1.77,73220,65419,71910,82683,79095,85219,83503,89943,90514,83445,81702,84259,81652,1052564,2.08
7,2-WEEKLY,3. ONLY RHA,182815,162704,174636,206083,195963,211039,210052,220538,224357,210760,200928,204687,196537,2601099,2.78,147089,131852,141554,166190,157877,170245,169438,178241,179455,169095,161142,163809,157583,2093570,2.88,110377,96424,104211,123328,117558,128612,126685,135114,135821,125649,121572,124993,118971,1569315,3.10
8,2-WEEKLY,4. NONE,138460,131737,142365,161485,155476,162821,160400,167932,173648,161303,155252,157419,150319,2018617,2.16,109168,103719,112080,126829,122595,128191,126605,132492,135851,126599,122099,123123,118047,1587398,2.19,84287,77837,84522,96519,93502,99137,97654,103526,105823,97019,95183,96948,91780,1223737,2.42
9,2-WEEKLY,5. IOS,624096,461736,503050,707662,643650,731803,712086,765377,799304,726867,700553,737190,697310,8810684,9.43,403545,304943,333740,454680,417004,471073,464339,488333,499754,459546,446056,467516,442695,5653224,7.79,329521,239963,262550,368994,339341,390445,380399,407414,419522,380336,372380,394450,369803,4655118,9.19


In [24]:
cx_data13.to_clipboard(index=False)

In [21]:
cx_data04 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        -- segment,
        case 
        when RHA_Check = 'RHA' and Zwigato_check = 'Zwigato' then '1. BOTH'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Zwigato' then '2. ONLY Zwigato'
        when RHA_Check = 'RHA' and Zwigato_check = 'Non-Zwigato' then '3. ONLY RHA'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Non-Zwigato' then '4. NONE'
        ELSE '5. IOS' end tech_savvy,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1
'''
cx_data14 = pd.read_sql(cx_data04, conn1)
cx_data14.head()

,tech_savvy,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,1. BOTH,2750562,2284771,2437729,2881957,2759161,3040833,3024866,3272108,3420549,3194638,3178082,3298705,3284273,38828234,41.57,2303834,1920224,2046625,2401887,2306184,2534648,2526902,2727355,2822916,2643383,2636443,2727429,2720248,32318078,44.51,1663363,1318116,1416318,1718293,1640472,1854862,1826348,2009113,2071471,1896431,1913072,2024249,1982959,23335067,46.08
1,2. ONLY Zwigato,475918,439124,472950,514521,503416,520775,521342,574857,597446,563671,565586,583837,599977,6933420,7.42,389029,359558,385763,417937,408843,424256,424482,468191,482178,455934,459237,472349,485293,5633050,7.76,261037,233790,254553,277452,272009,286113,282612,318145,325975,302778,307311,323004,328502,3773281,7.45
2,3. ONLY RHA,809041,743002,769000,847885,843000,888157,886311,958457,993120,939791,917541,941728,958322,11495355,12.31,661105,610266,630388,692103,689202,724153,723226,781988,802658,760428,744265,762358,775935,9358075,12.89,410196,365649,380839,423770,419049,451014,445771,492130,502678,466096,462550,483887,486746,5790375,11.44
3,4. NONE,828781,789804,830341,903547,913230,920278,914901,996603,1054384,989839,982941,1010715,1066795,12202159,13.06,658038,628775,659035,713981,724213,728145,725175,789039,828012,779041,775175,794101,836545,9639275,13.27,398605,369281,389336,425314,429395,439303,433661,482773,503836,465407,468921,490139,509072,5805043,11.46
4,5. IOS,1682043,1281654,1370981,1791316,1664099,1896499,1874552,2072889,2188983,2014663,1962298,2084254,2058312,23942543,25.63,1112897,866087,927635,1175008,1101179,1247070,1245459,1346725,1398521,1300274,1272417,1345822,1328498,15667592,21.58,840769,625288,672830,883029,827142,956649,945737,1040453,1083114,991494,983454,1054128,1027225,11931312,23.56


In [22]:
cx_data14

,tech_savvy,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,1. BOTH,2750562,2284771,2437729,2881957,2759161,3040833,3024866,3272108,3420549,3194638,3178082,3298705,3284273,38828234,41.57,2303834,1920224,2046625,2401887,2306184,2534648,2526902,2727355,2822916,2643383,2636443,2727429,2720248,32318078,44.51,1663363,1318116,1416318,1718293,1640472,1854862,1826348,2009113,2071471,1896431,1913072,2024249,1982959,23335067,46.08
1,2. ONLY Zwigato,475918,439124,472950,514521,503416,520775,521342,574857,597446,563671,565586,583837,599977,6933420,7.42,389029,359558,385763,417937,408843,424256,424482,468191,482178,455934,459237,472349,485293,5633050,7.76,261037,233790,254553,277452,272009,286113,282612,318145,325975,302778,307311,323004,328502,3773281,7.45
2,3. ONLY RHA,809041,743002,769000,847885,843000,888157,886311,958457,993120,939791,917541,941728,958322,11495355,12.31,661105,610266,630388,692103,689202,724153,723226,781988,802658,760428,744265,762358,775935,9358075,12.89,410196,365649,380839,423770,419049,451014,445771,492130,502678,466096,462550,483887,486746,5790375,11.44
3,4. NONE,828781,789804,830341,903547,913230,920278,914901,996603,1054384,989839,982941,1010715,1066795,12202159,13.06,658038,628775,659035,713981,724213,728145,725175,789039,828012,779041,775175,794101,836545,9639275,13.27,398605,369281,389336,425314,429395,439303,433661,482773,503836,465407,468921,490139,509072,5805043,11.46
4,5. IOS,1682043,1281654,1370981,1791316,1664099,1896499,1874552,2072889,2188983,2014663,1962298,2084254,2058312,23942543,25.63,1112897,866087,927635,1175008,1101179,1247070,1245459,1346725,1398521,1300274,1272417,1345822,1328498,15667592,21.58,840769,625288,672830,883029,827142,956649,945737,1040453,1083114,991494,983454,1054128,1027225,11931312,23.56


In [23]:
cx_data14.to_clipboard(index=False)

### regularity_segment

In [39]:
cx_data = f'''
    with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    
    select
        taxi_regularity_segment regularity_segment,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data1 = pd.read_sql(cx_data, conn3)
cx_data1.head()

,regularity_segment,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,MONTHLY,651285,613551,637556,676078,663779,663093,637160,678031,681770,604298,579957,632321,689274,8408153,9.00,523326,496021,514116,540219,532670,530251,510288,540432,538878,478953,461912,499036,544507,6710609,9.24,305060,279754,289352,302315,295743,299062,282338,307401,302548,258334,249342,283879,312160,3767288,7.44
1,UNKNOWN,355265,339707,357322,384858,399291,418663,428405,489018,535700,516656,526002,554868,624655,5930410,6.35,286639,276302,289484,309349,321772,336388,344930,391397,424240,410819,419674,437748,488580,4737322,6.52,119513,109986,116802,125386,131001,141888,143541,171710,187624,175254,183020,198670,225739,2030134,4.01
2,RARE_NEED,631500,456054,394384,352272,302581,276878,262987,276041,285614,265061,260419,258633,308638,4331062,4.64,509535,373714,324079,289566,251889,230641,219659,229356,235703,219499,216045,213326,251447,3564459,4.91,275822,178541,145673,122995,101038,89637,83700,89922,92201,82293,82327,82109,104416,1530674,3.02
3,DAILY,1916379,1504752,1646840,2233051,2095439,2415072,2411485,2590774,2704660,2528311,2435542,2540897,2399189,29422391,31.50,1458037,1160254,1272910,1691209,1594765,1829135,1838863,1958642,2014295,1899383,1836343,1906585,1803218,22263639,30.66,1242808,960589,1057683,1430403,1346690,1567682,1558042,1685248,1734444,1609403,1576203,1653003,1545973,18968171,37.46
4,BI_WEEKLY,1168344,1055949,1123839,1240899,1225475,1285922,1269416,1409528,1499074,1427071,1466936,1523066,1537596,17233115,18.45,927988,845166,897295,982184,974148,1019035,1007837,1112515,1170756,1117877,1151085,1192597,1205816,13604299,18.73,598562,522251,556818,612738,605057,642204,624313,705184,739109,691393,728823,779377,780731,8586560,16.96


In [47]:
cx_data1

,regularity_segment,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,MONTHLY,651285,613551,637556,676078,663779,663093,637160,678031,681770,604298,579957,632321,689274,8408153,9.00,523326,496021,514116,540219,532670,530251,510288,540432,538878,478953,461912,499036,544507,6710609,9.24,305060,279754,289352,302315,295743,299062,282338,307401,302548,258334,249342,283879,312160,3767288,7.44
1,UNKNOWN,355265,339707,357322,384858,399291,418663,428405,489018,535700,516656,526002,554868,624655,5930410,6.35,286639,276302,289484,309349,321772,336388,344930,391397,424240,410819,419674,437748,488580,4737322,6.52,119513,109986,116802,125386,131001,141888,143541,171710,187624,175254,183020,198670,225739,2030134,4.01
2,RARE_NEED,631500,456054,394384,352272,302581,276878,262987,276041,285614,265061,260419,258633,308638,4331062,4.64,509535,373714,324079,289566,251889,230641,219659,229356,235703,219499,216045,213326,251447,3564459,4.91,275822,178541,145673,122995,101038,89637,83700,89922,92201,82293,82327,82109,104416,1530674,3.02
3,DAILY,1916379,1504752,1646840,2233051,2095439,2415072,2411485,2590774,2704660,2528311,2435542,2540897,2399189,29422391,31.50,1458037,1160254,1272910,1691209,1594765,1829135,1838863,1958642,2014295,1899383,1836343,1906585,1803218,22263639,30.66,1242808,960589,1057683,1430403,1346690,1567682,1558042,1685248,1734444,1609403,1576203,1653003,1545973,18968171,37.46
4,BI_WEEKLY,1168344,1055949,1123839,1240899,1225475,1285922,1269416,1409528,1499074,1427071,1466936,1523066,1537596,17233115,18.45,927988,845166,897295,982184,974148,1019035,1007837,1112515,1170756,1117877,1151085,1192597,1205816,13604299,18.73,598562,522251,556818,612738,605057,642204,624313,705184,739109,691393,728823,779377,780731,8586560,16.96
5,WEEKLY,1823572,1568342,1721060,2052068,1996341,2206914,2212519,2431522,2547664,2361205,2337592,2409454,2408327,28076580,30.06,1419378,1233453,1351562,1588389,1554377,1712822,1723667,1880956,1950413,1812529,1802478,1852767,1852951,21735742,29.93,1032205,861003,947548,1134021,1108538,1247468,1242195,1383149,1431148,1305529,1315593,1378369,1365485,15752251,31.11


In [15]:
cx_data1.to_clipboard()

### ps_tag_link

In [157]:
cx_data2 = f'''
    with base as (
    select 
        customer_id,
        ps_tag_link
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317_link
    )
    
    
    select
        ps_tag_link ps_tag_link,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct
        
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data2 = pd.read_sql(cx_data2, conn4)
cx_data2.head()

,ps_tag_link,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,UNKNOWN,5275091,4383739,4597658,5400129,5240639,5765750,5739021,6276479,6596474,6197152,6123607,6384666,6455279,74435684,80.44,3830170,3140096,3328991,3865976,3706911,4018051,4073649,4392970,4686094,4453918,4383640,4603650,4660082,53144198,79.36,683119,528641,569447,679319,651527,740978,729134,804271,828170,777461,801611,856911,853434,9504023,61.98
1,PS,528023,477447,525654,634242,607480,650661,645349,697568,725917,661102,647769,666194,655413,8122819,8.78,411116,359889,402529,492133,459499,485442,490545,527040,562529,517226,501947,523406,515818,6249119,9.33,129517,105976,119594,151876,140360,155525,151222,164515,169481,150979,154658,164099,156452,1914254,12.48
2,NPS,668345,612158,678130,831411,762053,791565,778480,845727,870223,782474,769276,797920,785675,9973437,10.78,514305,455042,512166,636344,568934,584191,584477,633415,665692,602260,588824,619560,610753,7575963,11.31,261448,225116,255125,329838,295576,323538,310060,341701,341189,301473,305308,320244,306070,3916686,25.54


In [158]:
cx_data2

,ps_tag_link,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,UNKNOWN,5275091,4383739,4597658,5400129,5240639,5765750,5739021,6276479,6596474,6197152,6123607,6384666,6455279,74435684,80.44,3830170,3140096,3328991,3865976,3706911,4018051,4073649,4392970,4686094,4453918,4383640,4603650,4660082,53144198,79.36,683119,528641,569447,679319,651527,740978,729134,804271,828170,777461,801611,856911,853434,9504023,61.98
1,PS,528023,477447,525654,634242,607480,650661,645349,697568,725917,661102,647769,666194,655413,8122819,8.78,411116,359889,402529,492133,459499,485442,490545,527040,562529,517226,501947,523406,515818,6249119,9.33,129517,105976,119594,151876,140360,155525,151222,164515,169481,150979,154658,164099,156452,1914254,12.48
2,NPS,668345,612158,678130,831411,762053,791565,778480,845727,870223,782474,769276,797920,785675,9973437,10.78,514305,455042,512166,636344,568934,584191,584477,633415,665692,602260,588824,619560,610753,7575963,11.31,261448,225116,255125,329838,295576,323538,310060,341701,341189,301473,305308,320244,306070,3916686,25.54


In [18]:
cx_data2.to_clipboard(index=False)

### ps_tag_auto

In [163]:
cx_data3 = f'''
    with base as (
    select 
        customer_id,
        ps_tag_auto
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317_auto
    )
    
    
    select
        ps_tag_auto ps_tag_auto,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data3 = pd.read_sql(cx_data3, conn4)
cx_data3.head()

,ps_tag_auto,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,PS,1861049,1644809,1741417,2040866,1948246,2013963,1892200,2001728,2049217,1872838,1814885,1860027,1812600,24553845,26.60,1413362,1265957,1342409,1544100,1484661,1532030,1452058,1525199,1541549,1415618,1375311,1404119,1370617,18666990,26.13,587459,519532,540107,611770,587859,592898,546026,585894,590054,539723,530899,555888,542925,7331034,25.01
1,UNKNOWN,1945421,1565960,1666914,1970822,1937635,2360791,2612244,3010474,3261651,3120840,3137581,3328004,3479330,33397667,36.18,1547059,1258558,1332127,1554133,1533292,1855208,2051465,2348487,2518591,2418872,2440072,2573591,2689460,26120915,36.57,423904,301216,322340,409689,412879,548967,631548,755643,815079,773551,791120,863051,896489,7945476,27.10
2,NPS,2665568,2260566,2395680,2851246,2719383,2810169,2632350,2777425,2851880,2620875,2559824,2633504,2577621,34356091,37.22,2086442,1787750,1894300,2216026,2124018,2179889,2048391,2146099,2175579,2010122,1971204,2019757,1980274,26639851,37.30,1066386,842457,898147,1121447,1055471,1155168,1104071,1180223,1194109,1099593,1084702,1134808,1101016,14037598,47.89


In [167]:
cx_data3

,ps_tag_auto,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,PS,1861049,1644809,1741417,2040866,1948246,2013963,1892200,2001728,2049217,1872838,1814885,1860027,1812600,24553845,26.60,1413362,1265957,1342409,1544100,1484661,1532030,1452058,1525199,1541549,1415618,1375311,1404119,1370617,18666990,26.13,587459,519532,540107,611770,587859,592898,546026,585894,590054,539723,530899,555888,542925,7331034,25.01
1,UNKNOWN,1945421,1565960,1666914,1970822,1937635,2360791,2612244,3010474,3261651,3120840,3137581,3328004,3479330,33397667,36.18,1547059,1258558,1332127,1554133,1533292,1855208,2051465,2348487,2518591,2418872,2440072,2573591,2689460,26120915,36.57,423904,301216,322340,409689,412879,548967,631548,755643,815079,773551,791120,863051,896489,7945476,27.10
2,NPS,2665568,2260566,2395680,2851246,2719383,2810169,2632350,2777425,2851880,2620875,2559824,2633504,2577621,34356091,37.22,2086442,1787750,1894300,2216026,2124018,2179889,2048391,2146099,2175579,2010122,1971204,2019757,1980274,26639851,37.30,1066386,842457,898147,1121447,1055471,1155168,1104071,1180223,1194109,1099593,1084702,1134808,1101016,14037598,47.89


In [13]:
cx_data3.to_clipboard(index=False)

### taxi_income_segment

In [63]:
cx_data4 = f'''
    with base as (
    select 
        customer_id,
        taxi_income_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    
    select
        taxi_income_segment taxi_income_segment,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data4 = pd.read_sql(cx_data4, conn3)
cx_data4.head()

,taxi_income_segment,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,HIGH_INCOME,3261022,2702366,2877364,3454105,3283714,3621802,3595337,3920038,4103063,3823338,3767373,3930552,3908992,46249066,49.52,2536177,2128105,2262540,2668162,2551665,2799544,2792738,3018281,3120906,2921187,2889785,3002099,2990162,35681351,49.14,1824584,1459946,1563515,1902659,1811230,2040164,2012150,2213219,2284351,2097271,2098316,2222182,2180445,25710032,50.78
1,MEDIUM_INCOME,1492483,1351461,1418422,1578184,1556863,1636600,1623990,1759905,1835704,1716694,1699790,1749783,1785324,21205203,22.70,1229569,1114332,1167423,1296002,1279510,1344433,1335928,1448074,1498154,1405098,1394687,1430559,1457472,17401241,23.96,798117,691409,732013,829720,815272,877071,862089,955558,981993,900394,907147,952457,955490,11258730,22.24
2,LOW_INCOME,278662,260056,271562,298849,301666,303748,300469,325376,345721,320742,313855,321977,339160,3981843,4.26,222551,207125,216490,237265,240132,241548,239462,260196,273741,254282,250042,255322,268023,3166179,4.36,138106,123657,130190,144850,145056,149871,146856,163424,170661,154878,154815,161123,166685,1950172,3.85
3,UNKNOWN,1514178,1224472,1313653,1608088,1540663,1704392,1702176,1869595,1969994,1841828,1825430,1916927,1934203,21965599,23.52,1136606,935348,1002993,1199487,1158314,1272747,1277116,1386747,1441484,1358493,1353023,1414079,1430862,16367299,22.54,813163,637112,688158,850629,816509,920835,913034,1010413,1050069,969663,975030,1039645,1031884,11716144,23.14


In [101]:
cx_data4

,taxi_income_segment,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,HIGH_INCOME,3261022,2702366,2877364,3454105,3283714,3621802,3595337,3920038,4103063,3823338,3767373,3930552,3908992,46249066,49.52,2536177,2128105,2262540,2668162,2551665,2799544,2792738,3018281,3120906,2921187,2889785,3002099,2990162,35681351,49.14,1824584,1459946,1563515,1902659,1811230,2040164,2012150,2213219,2284351,2097271,2098316,2222182,2180445,25710032,50.78
1,MEDIUM_INCOME,1492483,1351461,1418422,1578184,1556863,1636600,1623990,1759905,1835704,1716694,1699790,1749783,1785324,21205203,22.70,1229569,1114332,1167423,1296002,1279510,1344433,1335928,1448074,1498154,1405098,1394687,1430559,1457472,17401241,23.96,798117,691409,732013,829720,815272,877071,862089,955558,981993,900394,907147,952457,955490,11258730,22.24
2,LOW_INCOME,278662,260056,271562,298849,301666,303748,300469,325376,345721,320742,313855,321977,339160,3981843,4.26,222551,207125,216490,237265,240132,241548,239462,260196,273741,254282,250042,255322,268023,3166179,4.36,138106,123657,130190,144850,145056,149871,146856,163424,170661,154878,154815,161123,166685,1950172,3.85
3,UNKNOWN,1514178,1224472,1313653,1608088,1540663,1704392,1702176,1869595,1969994,1841828,1825430,1916927,1934203,21965599,23.52,1136606,935348,1002993,1199487,1158314,1272747,1277116,1386747,1441484,1358493,1353023,1414079,1430862,16367299,22.54,813163,637112,688158,850629,816509,920835,913034,1010413,1050069,969663,975030,1039645,1031884,11716144,23.14


In [17]:
cx_data4.to_clipboard(index=False)

### taxi_service_affinity

In [67]:
cx_data5 = f'''
    with base as (
    select 
        customer_id,
        taxi_service_affinity
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    select
        taxi_service_affinity taxi_service_affinity,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data5 = pd.read_sql(cx_data5, conn3)
cx_data5.head()

,taxi_service_affinity,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,ONLY_AUTO,2363143,1906300,2044636,2583279,2431540,2736296,2712746,2927571,3051982,2862293,2796083,2914891,2838717,34169477,36.58,1810440,1481085,1590441,1966602,1863981,2084859,2080346,2222861,2286521,2156875,2118455,2198054,2144696,26005216,35.81,1455025,1144050,1229868,1566164,1474135,1684804,1668392,1807780,1859495,1738476,1714529,1804385,1743363,20890466,41.26
1,ONLY_CAB,191246,164366,171371,220834,210157,239516,232789,247574,260661,240551,228958,238366,229781,2876170,3.08,140224,123572,128079,158378,153547,172225,168447,177908,185071,172205,165866,170831,165248,2081601,2.87,108692,90802,94960,121934,117072,136119,130768,140247,148364,134953,129572,135585,129765,1618833,3.20
2,NO_AFFINITY,622906,561354,602322,701339,694912,746481,746919,811510,847393,780412,772282,792255,774856,9454941,10.12,475508,432427,464868,534902,530700,569007,570749,616398,635354,588567,583698,597817,585051,7185046,9.89,369494,323973,349011,410017,405024,447420,442459,490218,506020,455132,458785,480126,462351,5600030,11.06
3,LINK_CAB,11363,10767,11706,12932,12844,13407,13454,14993,16209,15118,14603,14941,15439,177776,0.19,8949,8526,9219,10061,10161,10533,10613,11691,12439,11849,11324,11588,12025,138978,0.19,5494,5189,5633,6293,6325,6688,6610,7646,8134,7513,7302,7799,7948,88574,0.17
4,AUTO_CAB,145517,124483,127533,155384,151758,169113,167170,180709,191018,177815,169743,177079,171802,2109124,2.26,107084,93623,95827,112825,111603,122901,122541,130938,136713,128171,123762,127982,124719,1538689,2.12,85319,70964,72290,88711,86902,99147,97507,106140,112216,103052,99491,104518,100383,1226640,2.42


In [81]:
cx_data5

,taxi_service_affinity,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,ONLY_AUTO,2363143,1906300,2044636,2583279,2431540,2736296,2712746,2927571,3051982,2862293,2796083,2914891,2838717,34169477,36.58,1810440,1481085,1590441,1966602,1863981,2084859,2080346,2222861,2286521,2156875,2118455,2198054,2144696,26005216,35.81,1455025,1144050,1229868,1566164,1474135,1684804,1668392,1807780,1859495,1738476,1714529,1804385,1743363,20890466,41.26
1,ONLY_CAB,191246,164366,171371,220834,210157,239516,232789,247574,260661,240551,228958,238366,229781,2876170,3.08,140224,123572,128079,158378,153547,172225,168447,177908,185071,172205,165866,170831,165248,2081601,2.87,108692,90802,94960,121934,117072,136119,130768,140247,148364,134953,129572,135585,129765,1618833,3.20
2,NO_AFFINITY,622906,561354,602322,701339,694912,746481,746919,811510,847393,780412,772282,792255,774856,9454941,10.12,475508,432427,464868,534902,530700,569007,570749,616398,635354,588567,583698,597817,585051,7185046,9.89,369494,323973,349011,410017,405024,447420,442459,490218,506020,455132,458785,480126,462351,5600030,11.06
3,LINK_CAB,11363,10767,11706,12932,12844,13407,13454,14993,16209,15118,14603,14941,15439,177776,0.19,8949,8526,9219,10061,10161,10533,10613,11691,12439,11849,11324,11588,12025,138978,0.19,5494,5189,5633,6293,6325,6688,6610,7646,8134,7513,7302,7799,7948,88574,0.17
4,AUTO_CAB,145517,124483,127533,155384,151758,169113,167170,180709,191018,177815,169743,177079,171802,2109124,2.26,107084,93623,95827,112825,111603,122901,122541,130938,136713,128171,123762,127982,124719,1538689,2.12,85319,70964,72290,88711,86902,99147,97507,106140,112216,103052,99491,104518,100383,1226640,2.42
5,UNKNOWN,2014159,1735521,1787675,1880956,1871126,1919020,1918942,2143021,2273216,2140517,2160184,2241127,2441851,26527315,28.40,1629379,1416671,1454037,1519001,1516976,1550554,1551930,1721769,1809340,1708887,1728800,1781956,1935385,21324685,29.37,853305,691589,715117,747427,749245,782302,775086,894702,937256,859095,886261,944196,1029098,10864679,21.46
6,LINK_AUTO,254158,222261,244858,284599,276467,301257,298826,327119,342160,316750,314038,329311,326417,3838221,4.11,200740,176986,194473,224735,218869,238320,236897,257680,267024,247757,246451,257491,255453,3022876,4.16,147625,124471,138171,162282,157984,174696,171308,191219,196897,177906,180847,193849,189330,2206585,4.36
7,ONLY_LINK,943853,813303,890900,1099903,1034102,1141452,1131126,1222417,1271843,1169146,1150557,1211269,1168816,14248687,15.26,752579,652020,712502,874412,823784,909873,903721,974053,1001823,924749,909181,956340,923942,11318979,15.59,549016,461086,508826,625030,591380,656765,641999,704662,718692,646079,658521,704949,672266,8139271,16.07


### RHA Check

In [73]:
cx_data6 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /*case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  */
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check

        /*CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check*/

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(RHA_Check, 'UNKNOWN') RHA_Check,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1 
'''
cx_data6 = pd.read_sql(cx_data6, conn3)
cx_data6.head()

,RHA_Check,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,NRHA,1299539,1224420,1298222,1412379,1411869,1435383,1431100,1565871,1648354,1552144,1549973,1599231,1685491,19113976,20.46,1042815,984673,1040582,1127583,1129474,1147823,1145505,1253040,1307653,1234109,1236039,1270828,1337986,15258110,21.01,655890,600630,640548,699263,698763,721859,712605,796999,826862,766448,775752,813749,844510,9553878,18.87
1,RHA,3559589,3028032,3207371,3730060,3601782,3929371,3910953,4230002,4410986,4130040,4088279,4229568,4217431,50273464,53.82,2965600,2531135,2678161,3094646,2995443,3259651,3250526,3509295,3624028,3400795,3375082,3481230,3475637,41641229,57.34,2074664,1684167,1798336,2142867,2059679,2306743,2273025,2501985,2573991,2361443,2373176,2504378,2459458,29113912,57.50
2,UNKNOWN,1687217,1285903,1375408,1796787,1669255,1901788,1879919,2079041,2195142,2020418,1968196,2090440,2064757,24014271,25.71,1116488,869102,930703,1178687,1104704,1250798,1249213,1350963,1402604,1304156,1276416,1350001,1332896,15716731,21.64,843416,627327,674992,885728,829625,959339,948499,1043630,1086221,994315,986380,1057280,1030536,11967288,23.63


In [103]:
cx_data6

,RHA_Check,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,NRHA,1299539,1224420,1298222,1412379,1411869,1435383,1431100,1565871,1648354,1552144,1549973,1599231,1685491,19113976,20.46,1042815,984673,1040582,1127583,1129474,1147823,1145505,1253040,1307653,1234109,1236039,1270828,1337986,15258110,21.01,655890,600630,640548,699263,698763,721859,712605,796999,826862,766448,775752,813749,844510,9553878,18.87
1,RHA,3559589,3028032,3207371,3730060,3601782,3929371,3910953,4230002,4410986,4130040,4088279,4229568,4217431,50273464,53.82,2965600,2531135,2678161,3094646,2995443,3259651,3250526,3509295,3624028,3400795,3375082,3481230,3475637,41641229,57.34,2074664,1684167,1798336,2142867,2059679,2306743,2273025,2501985,2573991,2361443,2373176,2504378,2459458,29113912,57.50
2,UNKNOWN,1687217,1285903,1375408,1796787,1669255,1901788,1879919,2079041,2195142,2020418,1968196,2090440,2064757,24014271,25.71,1116488,869102,930703,1178687,1104704,1250798,1249213,1350963,1402604,1304156,1276416,1350001,1332896,15716731,21.64,843416,627327,674992,885728,829625,959339,948499,1043630,1086221,994315,986380,1057280,1030536,11967288,23.63


### Zwigato_check

In [77]:
cx_data7 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        */

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(Zwigato_check, 'UNKNOWN') Zwigato_check,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
'''
cx_data7 = pd.read_sql(cx_data7, conn3)
cx_data7.head()

,Zwigato_check,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,Non-Zwigato,1638395,1532804,1599820,1752002,1757587,1809236,1802770,1956814,2051590,1936180,1909319,1961064,2042988,23750569,25.43,1319762,1238980,1289928,1406831,1414580,1453203,1449850,1572759,1634072,1544778,1526694,1564166,1627668,19043271,26.22,808647,734879,770622,849539,849218,890681,880098,975908,1008420,934109,935256,978271,1004881,11620529,22.95
1,UNKNOWN,1687217,1285903,1375408,1796787,1669255,1901788,1879919,2079041,2195142,2020418,1968196,2090440,2064757,24014271,25.71,1116488,869102,930703,1178687,1104704,1250798,1249213,1350963,1402604,1304156,1276416,1350001,1332896,15716731,21.64,843416,627327,674992,885728,829625,959339,948499,1043630,1086221,994315,986380,1057280,1030536,11967288,23.63
2,Zwigato,3220733,2719648,2905773,3390437,3256064,3555518,3539283,3839059,4007750,3746004,3728933,3867735,3859934,45636871,48.86,2688653,2276828,2428815,2815398,2710337,2954271,2946181,3189576,3297609,3090126,3084427,3187892,3185955,37856068,52.13,1921907,1549918,1668262,1992591,1909224,2137921,2105532,2323076,2392433,2193782,2213672,2339856,2299087,27047261,53.42


In [105]:
cx_data7

,Zwigato_check,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,Non-Zwigato,1638395,1532804,1599820,1752002,1757587,1809236,1802770,1956814,2051590,1936180,1909319,1961064,2042988,23750569,25.43,1319762,1238980,1289928,1406831,1414580,1453203,1449850,1572759,1634072,1544778,1526694,1564166,1627668,19043271,26.22,808647,734879,770622,849539,849218,890681,880098,975908,1008420,934109,935256,978271,1004881,11620529,22.95
1,UNKNOWN,1687217,1285903,1375408,1796787,1669255,1901788,1879919,2079041,2195142,2020418,1968196,2090440,2064757,24014271,25.71,1116488,869102,930703,1178687,1104704,1250798,1249213,1350963,1402604,1304156,1276416,1350001,1332896,15716731,21.64,843416,627327,674992,885728,829625,959339,948499,1043630,1086221,994315,986380,1057280,1030536,11967288,23.63
2,Zwigato,3220733,2719648,2905773,3390437,3256064,3555518,3539283,3839059,4007750,3746004,3728933,3867735,3859934,45636871,48.86,2688653,2276828,2428815,2815398,2710337,2954271,2946181,3189576,3297609,3090126,3084427,3187892,3185955,37856068,52.13,1921907,1549918,1668262,1992591,1909224,2137921,2105532,2323076,2392433,2193782,2213672,2339856,2299087,27047261,53.42


### Office Usecases

In [85]:
cx_data8 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams|intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag    
        from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(office_tag, 'UNKNOWN') office_tag,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1 
'''
cx_data8 = pd.read_sql(cx_data8, conn3)
cx_data8.head()

,office_tag,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,UNKNOWN,1687217,1285903,1375408,1796787,1669255,1901788,1879919,2079041,2195142,2020418,1968196,2090440,2064757,24014271,25.71,1116488,869102,930703,1178687,1104704,1250798,1249213,1350963,1402604,1304156,1276416,1350001,1332896,15716731,21.64,843416,627327,674992,885728,829625,959339,948499,1043630,1086221,994315,986380,1057280,1030536,11967288,23.63
1,no-office-apps,3352239,3056492,3208011,3543904,3535654,3673497,3639118,3953839,4140513,3895090,3852842,3948104,4064386,47863689,51.24,2739088,2503510,2621597,2883114,2880050,2989548,2965004,3216754,3340448,3147384,3120982,3189047,3281207,38877733,53.54,1802499,1585642,1674934,1871895,1860413,1975086,1936505,2144648,2216519,2047155,2052790,2142420,2172765,25483271,50.33
2,office-apps,1506889,1195960,1297582,1598535,1477997,1691257,1702935,1842034,1918827,1787094,1785410,1880695,1838536,21523751,23.04,1269327,1012298,1097146,1339115,1244867,1417926,1431027,1545581,1591233,1487520,1490139,1563011,1532416,18021606,24.82,928055,699155,763950,970235,898029,1053516,1049125,1154336,1184334,1080736,1096138,1175707,1131203,13184519,26.04


In [89]:
cx_data8

,office_tag,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,UNKNOWN,1687217,1285903,1375408,1796787,1669255,1901788,1879919,2079041,2195142,2020418,1968196,2090440,2064757,24014271,25.71,1116488,869102,930703,1178687,1104704,1250798,1249213,1350963,1402604,1304156,1276416,1350001,1332896,15716731,21.64,843416,627327,674992,885728,829625,959339,948499,1043630,1086221,994315,986380,1057280,1030536,11967288,23.63
1,no-office-apps,3352239,3056492,3208011,3543904,3535654,3673497,3639118,3953839,4140513,3895090,3852842,3948104,4064386,47863689,51.24,2739088,2503510,2621597,2883114,2880050,2989548,2965004,3216754,3340448,3147384,3120982,3189047,3281207,38877733,53.54,1802499,1585642,1674934,1871895,1860413,1975086,1936505,2144648,2216519,2047155,2052790,2142420,2172765,25483271,50.33
2,office-apps,1506889,1195960,1297582,1598535,1477997,1691257,1702935,1842034,1918827,1787094,1785410,1880695,1838536,21523751,23.04,1269327,1012298,1097146,1339115,1244867,1417926,1431027,1545581,1591233,1487520,1490139,1563011,1532416,18021606,24.82,928055,699155,763950,970235,898029,1053516,1049125,1154336,1184334,1080736,1096138,1175707,1131203,13184519,26.04


### Age Group

In [91]:
cx_data9 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_session as int) as ao_session,
            cast(fe_session as int) as fe_session,
            cast(rr_session as int) as rr_session
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    age as (
    
    select
        customer_id,
        age_group
    from 
        hive.experiments_internal.customer_predicated_age_immutable
    )
    
    
    select
        coalesce(age_group, 'UNKNOWN') age_group,
        sum(case when week_start_date = '20241216' then ao_session end ) w01_ao,
        sum(case when week_start_date = '20241223' then ao_session end ) w02_ao,
        sum(case when week_start_date = '20241230' then ao_session end ) w03_ao,
        sum(case when week_start_date = '20250106' then ao_session end ) w04_ao,
        sum(case when week_start_date = '20250113' then ao_session end ) w05_ao,
        sum(case when week_start_date = '20250120' then ao_session end ) w06_ao,
        sum(case when week_start_date = '20250127' then ao_session end ) w07_ao,
        sum(case when week_start_date = '20250203' then ao_session end ) w08_ao,
        sum(case when week_start_date = '20250210' then ao_session end ) w09_ao,
        sum(case when week_start_date = '20250217' then ao_session end ) w10_ao,
        sum(case when week_start_date = '20250224' then ao_session end ) w11_ao,
        sum(case when week_start_date = '20250303' then ao_session end ) w12_ao,
        sum(case when week_start_date = '20250310' then ao_session end ) w13_ao,
        
        sum(orders.ao_session) as overall_ao,
        100.00*sum(ao_session)/sum(sum(ao_session))over() as pct,
        
        sum(case when week_start_date = '20241216' then fe_session end ) w01_fe,
        sum(case when week_start_date = '20241223' then fe_session end ) w02_fe,
        sum(case when week_start_date = '20241230' then fe_session end ) w03_fe,
        sum(case when week_start_date = '20250106' then fe_session end ) w04_fe,
        sum(case when week_start_date = '20250113' then fe_session end ) w05_fe,
        sum(case when week_start_date = '20250120' then fe_session end ) w06_fe,
        sum(case when week_start_date = '20250127' then fe_session end ) w07_fe,
        sum(case when week_start_date = '20250203' then fe_session end ) w08_fe,
        sum(case when week_start_date = '20250210' then fe_session end ) w09_fe,
        sum(case when week_start_date = '20250217' then fe_session end ) w10_fe,
        sum(case when week_start_date = '20250224' then fe_session end ) w11_fe,
        sum(case when week_start_date = '20250303' then fe_session end ) w12_fe,
        sum(case when week_start_date = '20250310' then fe_session end ) w13_fe,
        
        sum(orders.fe_session) as overall_fe,
        100.00*sum(fe_session)/sum(sum(fe_session))over() as pct,

        sum(case when week_start_date = '20241216' then rr_session end ) w01_rr,
        sum(case when week_start_date = '20241223' then rr_session end ) w02_rr,
        sum(case when week_start_date = '20241230' then rr_session end ) w03_rr,
        sum(case when week_start_date = '20250106' then rr_session end ) w04_rr,
        sum(case when week_start_date = '20250113' then rr_session end ) w05_rr,
        sum(case when week_start_date = '20250120' then rr_session end ) w06_rr,
        sum(case when week_start_date = '20250127' then rr_session end ) w07_rr,
        sum(case when week_start_date = '20250203' then rr_session end ) w08_rr,
        sum(case when week_start_date = '20250210' then rr_session end ) w09_rr,
        sum(case when week_start_date = '20250217' then rr_session end ) w10_rr,
        sum(case when week_start_date = '20250224' then rr_session end ) w11_rr,
        sum(case when week_start_date = '20250303' then rr_session end ) w12_rr,
        sum(case when week_start_date = '20250310' then rr_session end ) w13_rr,
        
        sum(orders.rr_session) as overall_rr,
        100.00*sum(rr_session)/sum(sum(rr_session))over() as pct

      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        age
        on base.customer_id = age.customer_id
    group by 1
    order by 1
'''
cx_data9 = pd.read_sql(cx_data9, conn3)
cx_data9.head(10)

,age_group,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,21-25,1233340,1093283,1184307,1316167,1313340,1356749,1355515,1463061,1482146,1345704,1349037,1355261,1391264,17239174,18.46,1023841,908548,983719,1088609,1088590,1123213,1123647,1212085,1218502,1109161,1112823,1115586,1144794,14253118,19.63,643507,545272,596644,673206,673127,711253,702401,775423,778274,696327,708091,730712,739772,8974009,17.72
1,26-30,1062138,931586,1003883,1139615,1081548,1168609,1170541,1267897,1284362,1174824,1172234,1209809,1202392,14869438,15.92,887539,779169,838231,947683,901389,973055,974478,1054108,1057997,969492,970245,998659,992773,12344818,17.00,601668,506845,547877,635503,602784,667050,659837,732572,735835,655536,668726,705658,687030,8406921,16.60
2,31-35,851805,744109,776805,902454,868952,944602,933311,999019,1006202,928132,904735,938997,930820,11729943,12.56,705533,618263,643429,742951,717334,777914,770678,824702,822778,761895,745043,769962,764598,9665080,13.31,498568,417855,437267,519212,498123,555092,544764,595025,595259,539997,535885,564438,550458,6851943,13.53
3,36-45,1227156,1068465,1105343,1279040,1252646,1357082,1347306,1431365,1422399,1331814,1282438,1317244,1304745,16727043,17.91,1006024,878011,905525,1042968,1022829,1107450,1102020,1170029,1156247,1084487,1048281,1073118,1063997,13660986,18.81,719117,594932,622845,738201,714334,795811,784433,846153,841565,779028,759678,791226,773883,9761206,19.28
4,Above-45,269005,229136,237912,278342,275500,300678,300386,321246,316843,297615,286214,292935,293865,3699677,3.96,220106,188214,195515,227090,224929,244805,245027,262525,257222,241795,233376,238124,239091,3017819,4.16,155194,126454,133765,159848,156570,174841,172453,188903,185998,172748,168331,174794,173205,2143104,4.23
5,Below-21,95658,85859,91464,100356,100912,103184,103006,110777,107600,97779,97584,97872,100012,1292063,1.38,78046,70120,74317,81727,81947,84011,83679,90168,87255,79478,79379,79561,81253,1050941,1.45,49795,43023,46445,51466,51169,53930,52812,58393,56246,50618,51189,52670,53095,670851,1.32
6,UNKNOWN,1807243,1385917,1481287,1923252,1790008,2035638,2011907,2281549,2634930,2526734,2514206,2707121,2744581,27844373,29.81,1203814,942585,1008710,1269888,1192603,1347824,1345715,1499681,1734284,1692752,1698390,1827049,1860013,18623308,25.65,906121,677743,729033,950422,891960,1029964,1017429,1146145,1293897,1227952,1243408,1355909,1357061,13827044,27.31


In [107]:
cx_data9

,age_group,w01_ao,w02_ao,w03_ao,w04_ao,w05_ao,w06_ao,w07_ao,w08_ao,w09_ao,w10_ao,w11_ao,w12_ao,w13_ao,overall_ao,pct,w01_fe,w02_fe,w03_fe,w04_fe,w05_fe,w06_fe,w07_fe,w08_fe,w09_fe,w10_fe,w11_fe,w12_fe,w13_fe,overall_fe,pct,w01_rr,w02_rr,w03_rr,w04_rr,w05_rr,w06_rr,w07_rr,w08_rr,w09_rr,w10_rr,w11_rr,w12_rr,w13_rr,overall_rr,pct
0,21-25,1233340,1093283,1184307,1316167,1313340,1356749,1355515,1463061,1482146,1345704,1349037,1355261,1391264,17239174,18.46,1023841,908548,983719,1088609,1088590,1123213,1123647,1212085,1218502,1109161,1112823,1115586,1144794,14253118,19.63,643507,545272,596644,673206,673127,711253,702401,775423,778274,696327,708091,730712,739772,8974009,17.72
1,26-30,1062138,931586,1003883,1139615,1081548,1168609,1170541,1267897,1284362,1174824,1172234,1209809,1202392,14869438,15.92,887539,779169,838231,947683,901389,973055,974478,1054108,1057997,969492,970245,998659,992773,12344818,17.00,601668,506845,547877,635503,602784,667050,659837,732572,735835,655536,668726,705658,687030,8406921,16.60
2,31-35,851805,744109,776805,902454,868952,944602,933311,999019,1006202,928132,904735,938997,930820,11729943,12.56,705533,618263,643429,742951,717334,777914,770678,824702,822778,761895,745043,769962,764598,9665080,13.31,498568,417855,437267,519212,498123,555092,544764,595025,595259,539997,535885,564438,550458,6851943,13.53
3,36-45,1227156,1068465,1105343,1279040,1252646,1357082,1347306,1431365,1422399,1331814,1282438,1317244,1304745,16727043,17.91,1006024,878011,905525,1042968,1022829,1107450,1102020,1170029,1156247,1084487,1048281,1073118,1063997,13660986,18.81,719117,594932,622845,738201,714334,795811,784433,846153,841565,779028,759678,791226,773883,9761206,19.28
4,Above-45,269005,229136,237912,278342,275500,300678,300386,321246,316843,297615,286214,292935,293865,3699677,3.96,220106,188214,195515,227090,224929,244805,245027,262525,257222,241795,233376,238124,239091,3017819,4.16,155194,126454,133765,159848,156570,174841,172453,188903,185998,172748,168331,174794,173205,2143104,4.23
5,Below-21,95658,85859,91464,100356,100912,103184,103006,110777,107600,97779,97584,97872,100012,1292063,1.38,78046,70120,74317,81727,81947,84011,83679,90168,87255,79478,79379,79561,81253,1050941,1.45,49795,43023,46445,51466,51169,53930,52812,58393,56246,50618,51189,52670,53095,670851,1.32
6,UNKNOWN,1807243,1385917,1481287,1923252,1790008,2035638,2011907,2281549,2634930,2526734,2514206,2707121,2744581,27844373,29.81,1203814,942585,1008710,1269888,1192603,1347824,1345715,1499681,1734284,1692752,1698390,1827049,1860013,18623308,25.65,906121,677743,729033,950422,891960,1029964,1017429,1146145,1293897,1227952,1243408,1355909,1357061,13827044,27.31


### Office Use-Cases

In [61]:
cx_data11 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    use_case as (
    
    select
        hex_12,
        primary_tag
    from 
        experiments_internal.combined_geo_usecase_hex_12_level
    where 
        current_city = 'Bangalore'
        and primary_tag = 'office'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        case 
        when pickup_location_hex_12 in (select distinct hex_12 from use_case) then 'office'
        when drop_location_hex_12 in (select distinct hex_12 from use_case) then 'office'
        else 'non-office' end office_use_case,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_snapshot
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2,3
    )
    
    select
        coalesce(office_use_case, 'non-office') office_use_case,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct
        
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
'''
cx_data11 = pd.read_sql(cx_data11, conn1)
cx_data11.head(10)

,office_use_case,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,non-office,3757198,3084644,3415534,3785345,3698478,4100249,4103338,4536983,4700796,4176593,4254353,4509548,4467628,52590687,84.19,2447354,2178490,2351068,2478663,2434847,2563861,2644707,2774820,2806257,2741052,2749270,2814135,2886677,33871201,86.27
1,office,678315,409165,441992,752293,627717,838591,777813,890011,961868,843056,815048,959335,877941,9873145,15.81,395052,278193,294443,408286,368179,426464,435475,453361,461871,462754,450412,482149,473975,5390614,13.73


In [62]:
cx_data11

,office_use_case,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,non-office,3757198,3084644,3415534,3785345,3698478,4100249,4103338,4536983,4700796,4176593,4254353,4509548,4467628,52590687,84.19,2447354,2178490,2351068,2478663,2434847,2563861,2644707,2774820,2806257,2741052,2749270,2814135,2886677,33871201,86.27
1,office,678315,409165,441992,752293,627717,838591,777813,890011,961868,843056,815048,959335,877941,9873145,15.81,395052,278193,294443,408286,368179,426464,435475,453361,461871,462754,450412,482149,473975,5390614,13.73
